# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked Actions & Reason Codes

Our XGBoost model outputs a probability of decline (`decline_risk_score`). We translate this into human-readable actions:

*   **`ml_decline_risk_high_imp`**: Assigned to pages with >50% decline probability AND >5,000 monthly impressions. **Action:** `High-Priority Review for Refresh`. (These are our biggest potential losses).
*   **`ml_decline_risk_standard`**: Assigned to pages with >50% decline probability but normal impressions. **Action:** `Review for Meta-Title Update`.

The queue is sorted descending by Impressions, ensuring editors protect the highest-traffic assets first.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended Use and Limits

**Intended Use:** This queue is a weekly **triage prioritization tool** for the SEO editorial team to decide which 20-50 pages require their immediate attention to prevent traffic decay.

**Limits:** The model only evaluates 4 numeric metrics (impressions, clicks, CTR, position). It is entirely blind to *content quality*, *search intent*, and *seasonality*. It cannot tell the difference between a failing page and a holiday-themed page naturally exiting its peak season.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human Review & The No-Go List

**Human Review Requirement:** Before executing an action, an editor MUST manually search the primary keyword to verify intent. If the keyword is a "Zero-Click" search (e.g., weather, time zone) where Google answers the question directly, the poor CTR is natural and the page should be ignored.

**The No-Go List (Never Automate):** We must NEVER pipe these model outputs directly into a Generative AI tool to automatically rewrite and publish meta-titles. The risk of brand damage or hallucinating irrelevant titles on high-traffic pages heavily outweighs the time saved.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring & Retrain Triggers

To ensure the model does not go stale, we will monitor the **False Positive Rate** of our weekly queues.

**Retrain Triggers:**
1.  **Algorithm Updates:** The model must be retrained from scratch immediately following any confirmed Google Core Algorithm Update, as these fundamentally alter how ranking signals correlate with traffic.
2.  **Accuracy Drift:** If the Precision of our flagged queue drops below 50% for two consecutive months, the model will be retrained on the most recent 60-day window.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [1]:
import duckdb
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
import os
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'

# 1. Quick Train of the Model (Feb data)
print("Training model...")
train_query = f"""
    WITH feb AS (SELECT content_hash_id, SUM(gsc_impressions) AS imp, SUM(gsc_clicks) AS clk, AVG(gsc_avg_position) AS pos, SUM(gsc_clicks)/SUM(gsc_impressions) AS ctr FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-02/*.parquet') GROUP BY 1 HAVING imp >= 100),
    mar AS (SELECT content_hash_id, SUM(gsc_impressions) AS imp_future FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet') GROUP BY 1)
    SELECT feb.*, CASE WHEN mar.imp_future < 0.8 * feb.imp THEN 1 ELSE 0 END AS is_declining FROM feb LEFT JOIN mar USING (content_hash_id)
"""
df_train = con.sql(train_query).df().fillna(0)
features = ['imp', 'clk', 'pos', 'ctr']
model = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
model.fit(df_train[features], df_train['is_declining'])

# 2. Pull the live queue (March data) to predict April actions
print("Scoring live queue...")
queue_query = f"""
    SELECT content_hash_id, SUM(gsc_impressions) AS imp, SUM(gsc_clicks) AS clk, AVG(gsc_avg_position) AS pos, SUM(gsc_clicks)/SUM(gsc_impressions) AS ctr
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY 1 HAVING imp >= 100
"""
df_queue = con.sql(queue_query).df().fillna(0)

# 3. Predict Risk Score
df_queue['decline_risk_score'] = model.predict_proba(df_queue[features])[:, 1]

# 4. Apply Action Logic
def assign_action(row):
    if row['decline_risk_score'] > 0.5:
        if row['imp'] > 5000: return 'ml_decline_risk_high_imp', 'High-Priority Review for Refresh'
        else: return 'ml_decline_risk_standard', 'Review for Meta-Title Update'
    return 'none', 'No Action Required'

df_queue[['reason_code', 'action']] = df_queue.apply(assign_action, axis=1, result_type='expand')

# Filter and sort by highest traffic at risk
df_actionable = df_queue[df_queue['action'] != 'No Action Required'].sort_values(by=['imp', 'decline_risk_score'], ascending=[False, False])

# 5. Export to CSV
os.makedirs('work/outputs', exist_ok=True)
df_actionable.to_csv('work/outputs/refresh_queue.csv', index=False)
print(f"\nSUCCESS! Exported {len(df_actionable)} flagged pages to work/outputs/refresh_queue.csv")

# 6. Export Figure for the Paper
os.makedirs('work/figures', exist_ok=True)
plt.figure(figsize=(6,6))
df_actionable['reason_code'].value_counts().plot.pie(autopct='%1.1f%%', colors=['#ff9999','#66b3ff'])
plt.title('Distribution of ML-Flagged Reason Codes')
plt.ylabel('')
plt.savefig('work/figures/action_mix.png', bbox_inches='tight')
print("SUCCESS! Exported action_mix.png to work/figures/")
plt.show()

ModuleNotFoundError: No module named 'duckdb'

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.